# MiniGrid RL

## Setup

In [ ]:
from pathlib import Path
import gymnasium as gym
from feature_extractor_simple import MinigridFeaturesExtractor
from levels import LevelOne, LevelTwo, ProceduralLevel
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
import pygame
from minigrid.wrappers import ImgObsWrapper
from tqdm import tqdm
import torch
from minigrid.wrappers import ImgObsWrapper, RGBImgPartialObsWrapper, OneHotPartialObsWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
import numpy as np


## Level 1

In [ ]:
n_envs = 16
n_timesteps = 12000
device = "cuda"
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 128},
    "normalize_images": False
}


def make_env():
    env = ProceduralLevel(render_mode="rgb_array", difficulty=1000, max_steps=100)
    env = OneHotPartialObsWrapper(env)
    env = ImgObsWrapper(env)
    return env

env = make_vec_env(
    make_env, 
    n_envs = n_envs, 
    vec_env_cls = SubprocVecEnv,
)

model_level_path = Path("models/one_hot_good_boy.zip")

if model_level_path.exists():
    model = RecurrentPPO.load(model_level_path, env=env,learning_rate=0.00005)
    print("Model loaded, continuing training")
    model.learn(total_timesteps = (n_timesteps * n_envs), progress_bar = True, reset_num_timesteps=False)
    model.save(model_level_path)
else:
    print("No saved model, Training a new model")
    model = RecurrentPPO(
        "CnnLstmPolicy", 
        env, 
        n_steps=128,
        batch_size=512,
        policy_kwargs = policy_kwargs, 
        verbose = 1, 
        device = device, 
    )
    model.learn(total_timesteps = (n_timesteps * n_envs), progress_bar = True)
    model.save(model_level_path)

In [ ]:
env = ProceduralLevel(render_mode="human", difficulty=1000, max_steps=)
env = OneHotPartialObsWrapper(env)
env = ImgObsWrapper(env)

obs, info = env.reset()
model_level_path = Path("models/one_hot_good_boy.zip")
model = RecurrentPPO.load(model_level_path, env=env)
lstm_states = None
num_envs = 1

episode_starts = np.ones((num_envs,), dtype=bool)

while True:
    action, lstm_states = model.predict(
        obs, 
        state=lstm_states, 
        episode_start=episode_starts,
        deterministic=True
    )
    
    obs, rewards, terminated, truncated, info = env.step(action)
    
    episode_starts = terminated or truncated
    
    if terminated or truncated:
        break
env.close()